# Forest Plot Table (`gt_forest_table`)

`gt_forest_table()` renders a [Great Tables](https://posit-dev.github.io/great-tables/) table where each row carries an inline forest plot — a CI whisker + point estimate dot + reference line — generated with matplotlib and embedded as a base64 PNG.

**When to use it:** whenever you want a publication-style summary table with a visual forest plot column alongside plain-text statistics (n, p-value, etc.) and hierarchical row labels produced by a `DataTree`.

**Key parameters**

| Parameter | Description |
|-----------|-------------|
| `x` | Stat column name for the **point estimate** |
| `xmin` | Stat column name for the **lower CI bound** |
| `xmax` | Stat column name for the **upper CI bound** |
| `info_columns` | `...` (default) = all other stat cols; `None` = none; `list[str]` = explicit selection |
| `ref_line` | X position of the vertical reference line (default `1.0`) |
| `autoscale` | Share x-axis range across all rows (default `True`) |
| `cascade` | Include structural tree nodes as rows (default `False`) |

## 1 — Imports and sample data

In [ ]:
import numpy as np
import pandas as pd
from pyMyriad import AnalysisTree, gt_forest_table

rng = np.random.default_rng(42)

n = 500
df = pd.DataFrame({
    "Treatment": rng.choice(["Drug A", "Drug B", "Placebo"], n),
    "Site":      rng.choice(["Site 1", "Site 2", "Site 3"], n),
    "Sex":       rng.choice(["M", "F"], n),
    # Simulated log-odds-ratio-like outcome (centred near 0)
    "logOR": rng.normal(loc=0.1, scale=0.8, size=n),
    "n_events": rng.integers(1, 20, n),
})
df.head()

## 2 — Basic forest table

The analysis tree produces three statistics named `x`, `xmin`, `xmax`.  
Pass those names to `gt_forest_table()` and you get an interactive GT table with one inline plot per row.

In [ ]:
def mean_ci(df, alpha=1.96):
    """Return point estimate and 95 % CI for mean logOR."""
    se = np.std(df.logOR, ddof=1) / np.sqrt(len(df))
    m  = np.mean(df.logOR)
    return m, m - alpha * se, m + alpha * se

tree_basic = (
    AnalysisTree()
    .split_by("df.Treatment")
    .analyze_by(
        x    = lambda df: mean_ci(df)[0],
        xmin = lambda df: mean_ci(df)[1],
        xmax = lambda df: mean_ci(df)[2],
        n    = lambda df: len(df),
    )
)

result_basic = tree_basic.run(df)

gt_forest_table(
    result_basic,
    x="x", xmin="xmin", xmax="xmax",
    ref_line=0.0,
    title="Mean log-OR by Treatment",
    subtitle="Error bars show 95 % CI",
)

## 3 — Controlling what extra columns appear (`info_columns`)

By default every stat column that isn't part of the CI triple is shown as plain text.  
Pass `info_columns=None` to suppress them, or supply a list to pick exact columns.

In [ ]:
# Only the forest plot column, no extra stats
gt_forest_table(
    result_basic,
    x="x", xmin="xmin", xmax="xmax",
    info_columns=None,
    ref_line=0.0,
    title="Plot only — no info columns",
)

In [ ]:
# Show only the 'n' column alongside the plot
gt_forest_table(
    result_basic,
    x="x", xmin="xmin", xmax="xmax",
    info_columns=["n"],
    ref_line=0.0,
    title="Plot + sample size only",
)

## 4 — Nested splits (two-level hierarchy)

Use multiple `split_by()` calls to produce a two-level table.  
Row labels are split across `Level 0` / `Level 1` columns automatically.

In [ ]:
tree_nested = (
    AnalysisTree()
    .split_by("df.Treatment")
    .split_by("df.Sex")
    .analyze_by(
        x    = lambda df: mean_ci(df)[0],
        xmin = lambda df: mean_ci(df)[1],
        xmax = lambda df: mean_ci(df)[2],
        n    = lambda df: len(df),
    )
)

result_nested = tree_nested.run(df)

gt_forest_table(
    result_nested,
    x="x", xmin="xmin", xmax="xmax",
    info_columns=["n"],
    ref_line=0.0,
    title="Mean log-OR by Treatment × Sex",
)

## 5 — `cascade=True`: show intermediate summary rows

Set `cascade=True` to include split/summary nodes in the table.  
Those rows have no CI values, so the Forest Plot cell is left blank — which is the correct visual treatment for structural rows.

In [ ]:
tree_cascade = (
    AnalysisTree()
    .split_by("df.Treatment")
    .summarize_by(
        x    = lambda df: mean_ci(df)[0],
        xmin = lambda df: mean_ci(df)[1],
        xmax = lambda df: mean_ci(df)[2],
        n    = lambda df: len(df),
    )
    .split_by("df.Site")
    .analyze_by(
        x    = lambda df: mean_ci(df)[0],
        xmin = lambda df: mean_ci(df)[1],
        xmax = lambda df: mean_ci(df)[2],
        n    = lambda df: len(df),
    )
)

result_cascade = tree_cascade.run(df)

gt_forest_table(
    result_cascade,
    x="x", xmin="xmin", xmax="xmax",
    cascade=True,
    info_columns=["n"],
    ref_line=0.0,
    title="Treatment summaries + Site breakdown",
    subtitle="Structural rows have blank forest plot cells",
)

## 6 — `autoscale=False`: per-row axis range

When `autoscale=False` each plot uses its own x-axis range.  
Useful when rows have very different magnitudes and visual comparison across rows is not the goal.

In [ ]:
gt_forest_table(
    result_basic,
    x="x", xmin="xmin", xmax="xmax",
    autoscale=False,
    ref_line=0.0,
    title="Per-row axis range (autoscale=False)",
)

## 7 — Mixed analyses: not every row has CI columns

When one `analyze_by` node produces `x`/`xmin`/`xmax` and another does not (e.g. a count-only node), the rows that lack CI data simply show a blank Forest Plot cell — no error is raised.

In [ ]:
tree_mixed = (
    AnalysisTree()
    .split_by("df.Treatment")
    .analyze_by(
        x    = lambda df: mean_ci(df)[0],
        xmin = lambda df: mean_ci(df)[1],
        xmax = lambda df: mean_ci(df)[2],
        label="CI",
    )
    .analyze_by(
        n = lambda df: len(df),
        label="Count",
    )
)

result_mixed = tree_mixed.run(df)

gt_forest_table(
    result_mixed,
    x="x", xmin="xmin", xmax="xmax",
    ref_line=0.0,
    title="Mixed analyses — Count rows have blank plot cells",
)